# 7. The clustering dataset, with every patient accounted for

**What it does**
1. Applies the inclusion rules in `CLUSTER_RULES` (in `sdh_config.py`) one at a time and counts who
   each one removes: has a CT report, SDH confirmed on CT, in the study period (+ optional: no
   elective, no ER-only code, no in-hospital onset).
2. Keeps the clustering inputs (group A).
3. Reports **missing data** per variable (overall and in-hospital onset separately), per hospital and
   per patient. **Nobody is dropped for missing labs or vitals:** a variable missing in more than
   40% of patients is set aside, and patients missing more than 50% are flagged (both thresholds
   are settings).

**Output:** `patient_level/clustering_dataset.csv` (join `outcomes.csv` on `row_id` for the outcome
analyses); aggregate flowchart and missing-data tables.
**Time:** seconds.

> **Privacy:** everything this notebook *displays* comes from `aggregate/` (counts with cells under 6 suppressed). Files in `patient_level/` stay on GEMINI: don't print their rows, and never paste them outside GEMINI.

In [ ]:
import os, sys
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

# To write the output somewhere else, uncomment and edit BEFORE importing sdh_config:
# os.environ["SDH_OUTPUT_DIR"] = str(Path.home() / "sdh_phenotype_output")

PHENO = Path.cwd().parent              # the phenotype/ folder (this notebook is in phenotype/notebooks/)
sys.path.insert(0, str(PHENO))
import sdh_config as cfg
import sdh_io as io
pd.set_option("display.max_rows", 200, "display.max_columns", 60, "display.width", 220)
AGG = cfg.AGGREGATE_DIR


def show(name, rows=None, note=None):
    """Display one aggregate (cell-suppressed) table from the output folder."""
    path = AGG / name
    if not path.exists():
        print(f"{name}: not there yet (run the step above first)")
        return None
    df = pd.read_csv(path)
    display(Markdown(f"**{name}**" + (f" - {note}" if note else "")))
    display(df.head(rows) if rows else df)
    return df


def share(names, rows=80):
    """Print aggregate tables as text to paste to Claude / your team (counts only, small cells suppressed)."""
    for name in names:
        path = AGG / name
        if path.exists():
            df = pd.read_csv(path)
            print(f"===== {name} ({len(df)} rows{', first ' + str(rows) if len(df) > rows else ''}) =====")
            print(df.head(rows).to_csv(index=False))
        else:
            print(f"===== {name}: not found =====")

## Settings in use

In [ ]:
print("Inclusion rules:", cfg.CLUSTER_RULES)
print("Set aside variables missing in more than", cfg.MAX_VARIABLE_MISSING_PCT, "%")
print("Flag patients missing more than", cfg.MAX_PATIENT_MISSING_PCT, "% | drop them:", cfg.DROP_HIGH_MISSING_PATIENTS)

To change a rule, edit `sdh_config.py`, then restart the kernel (Kernel -> Restart) and rerun.

## Run

In [ ]:
%run ../06_clustering_dataset.py

## Results

In [ ]:
show("clustering_flowchart.csv", note="the flowchart for the paper");

In [ ]:
show("clustering_missingness_by_variable.csv", note="sorted from most to least missing");

**Look for:** which variables are set aside (`used_for_clustering = False`); CTAS / mLAPS are blank for
in-hospital onset by design (see the split columns).

In [ ]:
show("clustering_patient_missingness.csv");

In [ ]:
show("clustering_missingness_by_hospital.csv");

**Look for:** a hospital missing a whole variable (data not sent) rather than scattered gaps.

## Share

In [ ]:
share(["clustering_flowchart.csv", "clustering_missingness_by_variable.csv", "clustering_patient_missingness.csv", "clustering_missingness_by_hospital.csv"])